# 02 — The multi-agent pipeline, end to end (and the clone-debate control)

```
 panels ─► EYE + RULER ─► 6 AGENTS ─► JUDGE ──(agree?)──► SUPERVISOR ───────────────────────────► rule
          (01; no fact     each reads   │                  1 clear measured rule, unless confident agents'
           from both)      its OWN      └─no─► DEBATE ─────┘   rule checks out instead
                           evidence            (1 round;   2 synthesis over round-1 + debate rules
                                               change only 3 check against the panel facts (≥ 11/12)
                                               with a named 4 next most agreed-on rules
                                               counter-    5 one look-again on the failing panels
                                               example)    6 a narrow measured rule, else the best-scoring
```

**Two agent styles, same everything else** (this is the fair test of debate):
- **specialist** — six agents that each read *different* evidence (below). Shape, topology, quantity, spatial and
  size each see only their own fields; the relational agent (the combiner) sees the objects, their relations and
  the ruler's comparison of the two largest figures.
- **clone** — six identical generalists that all read the full record: the setting in which prior work found
  debate does not help.

Every agent answers in the same structure — `has_rule, left, right, confidence, evidence` — so the judge, the
supervisor and the check compare rules, not prose. Agents with no rule abstain from consensus.
- **Evidence is passed on:** each agent's one-sentence evidence reaches the debate and the supervisor.
- **Debate guard:** in debate an agent keeps its rule unless it names a panel that breaks it. Every attempted
  switch is logged, for the conformity analysis.
- **Supervisor input:** the supervisor and the check read the panel facts in words (no raw measurements), plus a
  short summary of the measured differences that truly separate the sides.
- **Check ablation:** both the checked and the unchecked final rule are saved, so the check can be scored on its own.

Runs resume; each problem is saved as it ends.

In [1]:
%run ./00_common.ipynb

repo: /project/painting/local-projects-to-run/AnacodicAI-Labs/code/production-repos/ai-in-foundations-and-methods/research_bongard_problems
keys: groq=True openai=True | vision=gpt-4o | agents=openai/gpt-oss-120b | grader=openai/gpt-oss-120b
100 classic problems | 21 development problems | sets: ['SMOKE', 'DEV_21', 'HELDOUT2', 'TEST_30', 'ALL_100']
00_common loaded.


In [2]:
PROBLEMS = PROBLEM_SETS[os.environ.get("PROBLEM_SET", "SMOKE")]
ALL_RUNS = [("specialist", "openai/gpt-oss-120b"), ("clone", "openai/gpt-oss-120b"), ("specialist", "openai/gpt-oss-20b")]
RUNS = ALL_RUNS if os.environ.get("ARMS", "main") == "all" else ALL_RUNS[:1]   # while testing, only the main arm
RERUN = False              # True = recompute problems that are already saved
MAX_DEBATE_ROUNDS = 1      # the original's max_debate_rounds=3 ran exactly 1 debate round; kept explicit here
USE_LLM_JUDGE = True       # original design: an LLM judge decides consensus; strong local consensus overrides it
print(len(PROBLEMS), "problems ×", len(RUNS), "arms")

21 problems × 1 arms


## Who reads what

In [3]:
ROUTES = {   # eye object attributes · eye fields · ruler measurements   (no fact comes from both)
    "shape":    dict(obj=("type", "sides", "fill"), keys=("symmetric",), ruler=("convexity", "corners", "angles")),
    "topology": dict(obj=("type",), keys=("relations", "lines"), ruler=("topology",)),
    "quantity": dict(obj=("type", "fill"), keys=("count", "lines"), ruler=("count",)),
    "spatial":  dict(obj=("type", "pointing"), keys=(), ruler=("position", "elongation")),
    "size":     dict(obj=("type",), keys=(), ruler=("size", "elongation")),
    "relation": dict(obj=("type", "fill"), keys=("relations",), ruler=("rank",)),
}
RULER_ATTRS = ("area_frac", "large", "small", "convexity", "compactness", "ncorners", "elongation", "aspect",
               "centroid_x", "centroid_y", "object_count")          # contrast only over what the ruler owns
CONTRAST_FOR = {"shape": ("convexity", "compactness", "ncorners"), "topology": (), "quantity": ("object_count",),
                "spatial": ("centroid_x", "centroid_y", "aspect"), "size": ("area_frac", "large", "small", "elongation", "aspect"),
                "relation": ()}
GUIDE = {   # adapted from the original specialists' dimension_instructions
    "shape": "Priority: fill (solid vs outline); then shape type and number of sides (triangles vs quadrangles, circles vs non-circles, curved vs straight-edged); then corners (acute or right-angle corners, measured); then convexity (measured convexity near 1 = convex, below 0.9 = has dents); then symmetry.",
    "topology": "If 'inside' (or touching, overlapping, or a line crossing itself) holds on all left panels and on no right panel, that is a clear topology rule. The seen relations come first; the measured box check is a rough second opinion.",
    "quantity": "Read the seen counts. The measured figure count is exact but merges touching shapes. If all left panels share one count and all right panels another (e.g. one vs two), that is the rule; if the count is the same on all 12 panels, count is NOT the rule.",
    "spatial": "Measured positions are exact (x and y from 0 to 1): below 0.45 vs above 0.55 suggests left side vs right side; the same with y for top vs bottom. The long axis gives vertical vs horizontal; pointing is seen.",
    "size": "Measured sizes are exact. A clear gap between the sides means large vs small; elongation and long-axis direction give elongated vs compact and vertical vs horizontal.",
    "relation": "Compare the objects inside each panel: which is bigger, which is above, which is inside. The measured comparison names each figure by its outline (3-corner, 4-corner, round). The rule is often a relation that flips between the two sides.",
}
FOCUS = {
    "shape": "shape (form type, number of sides, fill, corner angles, symmetry, convexity)",
    "topology": "topology (containment, touching, overlapping, lines crossing, connectivity)",
    "quantity": "quantity (number of objects, number of lines)",
    "spatial": "spatial (position, pointing direction, long-axis direction)",
    "size": "size (absolute and relative sizes, elongation)",
    "relation": "relations BETWEEN the objects in the same panel (which is bigger, above or below, inside, same or different)",
}

def routed_blocks(bp, agent):
    """The evidence one specialist is allowed to read, for all 12 panels."""
    r, f, cv = ROUTES[agent], vlm_fields(bp), load_cv(bp)
    recs = {}
    for pid in PIDS:
        s = eye_text(f.get(pid), obj=r["obj"], keys=r["keys"])
        if r["ruler"]:
            s += " || MEASURED: " + ruler_text(cv[pid], keys=r["ruler"])
        recs[pid] = s
    return records_block(recs)

left, right = routed_blocks("bp0038" if "bp0038" in PROBLEMS else PROBLEMS[0], "relation")
print("what the RELATION specialist reads (first 2 left panels):\n" + "\n".join(left.splitlines()[:2]))

what the RELATION specialist reads (first 2 left panels):
  L1: objects: outline triangle; outline circle | relations: circle to the right of triangle || MEASURED: the 3-corner figure is clearly bigger than the round figure and sits left of it
  L2: objects: outline triangle; outline circle | relations: circle inside triangle || MEASURED: the 3-corner figure is clearly bigger than the round figure and sits at the same place as it


## Prompts and answer handling

In [4]:
STYLE = ("Say each side as a short visual concept a person would use (3-10 words), e.g. 'large figures', 'a triangle above a circle', 'a line that crosses itself'. "
         "Name what IS on each side, not what is missing ('triangles', not 'no quadrilaterals'), unless absence itself is the rule. "
         "Do NOT mention numbers, thresholds, coordinates, 'ruler', 'blob', 'eye', 'record' or panel labels.")
ANSWER = (STYLE + '\n' + 'Return JSON only: {{"has_rule": true, "left": "<what all 6 left panels share>", '
          '"right": "<what all 6 right panels share>", "confidence": 0.0, "evidence": "<one sentence>"}}')

AGENT_PROMPT = """You are {who}.
{focus}
This is a Bongard problem: the 6 LEFT panels all follow ONE hidden rule; the 6 RIGHT panels break it (usually they show the opposite).
Evidence for each panel:
LEFT panels:
{left}
RIGHT panels:
{right}
{hint}
Find the simplest rule that is true for ALL 6 left panels and false for ALL 6 right panels, using this evidence.
Check it panel by panel (L1..L6 must fit, R1..R6 must not). Prefer simple, human-readable rules.
A reading can be wrong on one panel: if a rule fits 11 of the 12 panels, still propose it with lower confidence
and name the exception in your evidence. If the evidence does not separate the sides, set has_rule to false.
""" + ANSWER

REQUIRE_COUNTEREXAMPLE = True    # in debate an agent may drop its rule only by naming a panel that breaks it
DEBATE_PROMPT = """You are {who}.
{focus}
A Bongard problem: the 6 LEFT panels follow one hidden rule; the 6 RIGHT panels break it.

YOUR current answer: {own}

Rules proposed by the others, with their evidence:
{others}

Your evidence:
LEFT panels:
{left}
RIGHT panels:
{right}
{hint}
Reconsider:
  - Does your rule still hold for all 12 panels? Look for a panel in YOUR evidence that breaks it.
  - Can your rule COMBINE with another into a stronger rule that still fits all 12 panels?
Change or drop your rule ONLY if you can name a panel (L1..L6 or R1..R6) in YOUR evidence that breaks it, and put
that panel in "counterexample". Others disagreeing is not a reason to change. If you combine rules, write "combined".
If you had no rule, you may adopt one that your evidence supports.
""" + STYLE + """
Return JSON only: {{"has_rule": true, "left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>",
"confidence": 0.0, "evidence": "<one sentence>", "counterexample": "<panel label, 'combined', or empty>"}}"""

JUDGE_PROMPT = """You are the judge evaluating the agents' rules for a Bongard problem.
The 6 LEFT panels follow one hidden rule; the 6 RIGHT panels break it.

Rules this round:
{table}

Do at least three agents propose the same underlying rule, even if worded differently? How many distinct rules
are there? Which agent has the strongest, best-supported rule?
Return JSON only: {{"consensus": true, "distinct_rules": 0, "dominant": "<agent name>", "reasoning": "<one sentence>"}}"""

SUP_PROMPT = """You are the supervisor writing the final answer for a Bongard problem.
The 6 LEFT panels share one hidden rule; the 6 RIGHT panels break it.

Rules proposed by the agents (most agreed-on first; each with the agent's evidence):
{table}
{judge_pick}
{instr}

Panel facts (what the panels show, in words):
LEFT panels:
{left}
RIGHT panels:
{right}

{contrast}{cv_note}

Write the single best rule. You MUST give a concrete rule — never "no rule" or "unknown". If no proposed rule fits
all 12 panels, derive one from the panel facts. Prefer the most specific rule the facts support.
""" + STYLE + """
Return JSON only: {{"left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>", "confidence": 0.0}}"""
INSTR = {"voluntary": "The agents agreed; state the consensus rule precisely.",
         "forced": "The agents did not agree; weigh their rules against the panel facts."}

LOOK_AGAIN_PROMPT = """You are the supervisor of a Bongard problem: the 6 LEFT panels share one hidden rule; the 6 RIGHT panels break it.
Your rule was: LEFT = "{L}" ; RIGHT = "{R}".
A check against the panel facts found these panels do not fit:
{fails}

Panel facts:
LEFT panels:
{left}
RIGHT panels:
{right}

Look again at the panels that do not fit and write a rule that fits all 12 (a revised or a different rule).
You MUST give a concrete rule.
""" + STYLE + """
Return JSON only: {{"left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>", "confidence": 0.0}}"""

NAME_PROMPT = """A Bongard rule was found: LEFT = "{L}" ; RIGHT = "{R}".
One side is described by what is MISSING. Using the panel facts, say what the panels on that side DO show instead
(e.g. "triangles" vs "circles", not "triangles" vs "no triangles"). If absence itself is the concept (e.g. a line
that crosses itself vs lines that do not cross), keep the rule as it is.

Panel facts:
LEFT panels:
{left}
RIGHT panels:
{right}
""" + STYLE + """
Return JSON only: {{"left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>", "confidence": 0.0}}"""

CHECK_PROMPT = """Check a candidate Bongard rule against written facts about 12 panels.
Candidate rule: LEFT panels = "{L}" ; RIGHT panels = "{R}".

LEFT panels:
{left}
RIGHT panels:
{right}

For EACH panel decide whether its facts fit the LEFT side of the rule, the RIGHT side, or neither.
Reply json: {{"L1": "left|right|neither", "L2": "...", "L3": "...", "L4": "...", "L5": "...", "L6": "...",
"R1": "...", "R2": "...", "R3": "...", "R4": "...", "R5": "...", "R6": "..."}}"""

def agent_json(prompt, model, **kw):
    """An agent call that never crashes the pipeline: a failed agent abstains."""
    try:
        return text_json(prompt, model=model, **kw)
    except BudgetExceeded:
        raise
    except Exception as e:
        print(f"      agent call failed, abstaining: {type(e).__name__}: {str(e)[:80]}")
        return {"has_rule": False}

def fmt_answer(a, evidence=False):
    if not a["has_rule"]:
        return "no rule"
    s = f"Left = {a['left']}, Right = {a['right']} (confidence {a['confidence']:.2f})"
    return s + (f" — evidence: {a['evidence']}" if evidence and a.get("evidence") else "")

def local_consensus(hyps, high=0.95, low=0.50, group=3):
    """Deterministic judge over rule-bearing answers only."""
    ruled = {n: h for n, h in hyps.items() if h["has_rule"]}
    if not ruled:
        return False, None
    confs = {n: h["confidence"] for n, h in ruled.items()}
    dom = max(confs, key=confs.get)
    if confs[dom] >= high and all(h["confidence"] < low for n, h in hyps.items() if n != dom):
        return True, dom
    groups = []
    for n, h in ruled.items():
        key = (h["left"] + " | " + h["right"]).lower()
        for g in groups:
            ref = ruled[g[0]]
            if SequenceMatcher(None, key, (ref["left"] + " | " + ref["right"]).lower()).ratio() > 0.6:
                g.append(n); break
        else:
            groups.append([n])
    return max(len(g) for g in groups) >= group, dom

def strong_override(hyps, high=0.95, low=0.50):
    """Original route: one agent >= 0.95 confident and every other agent below 0.5 -> skip debate."""
    ruled = {n: h for n, h in hyps.items() if h["has_rule"]}
    if not ruled:
        return False
    dom = max(ruled, key=lambda n: ruled[n]["confidence"])
    return ruled[dom]["confidence"] >= high and all(h["confidence"] < low for n, h in hyps.items() if n != dom)

def check_rule(left_rule, right_rule, left, right, model):
    """Text-only check: how many of the 12 panels land on their own side of the rule, and which ones do not."""
    out = agent_json(CHECK_PROMPT.format(L=left_rule, R=right_rule or f"not {left_rule}", left=left, right=right),
                     model=model, max_tokens=1500)
    fails = []
    for side, want in (("L", "left"), ("R", "right")):
        for k in range(1, 7):
            got = str(out.get(f"{side}{k}", "")).lower()
            if not got.startswith(want):
                fails.append(f"{side}{k} fits {'the ' + got.upper() + ' side' if got in ('left', 'right') else 'neither side'}")
    return 12 - len(fails), fails

_NEGATED = re.compile(r"\b(no|not|without|lacks?|never|absent)\b|\bnon[\s\-\u2010\u2011]", re.I)
_JARGON = re.compile(r"\b(ruler|blobs?|eye|records?|measured|\d-corner|[LRP]\d{1,2})\b", re.I)

def negated(h):
    """A side described by what is missing ('no triangle', 'non-triangular')."""
    return bool(_NEGATED.search(h["left"]) or _NEGATED.search(h["right"]))

def jargon(h):
    """Rule text that talks about the pipeline's evidence instead of the picture."""
    return bool(_JARGON.search(h["left"] + " " + h["right"]))

def candidates(round1, final):
    """Round-1 and post-debate rules together, near-duplicates merged. Most agreed-on first, then rules that name
    both sides positively and without jargon, then confidence."""
    pool = []
    for stage in (round1, final):
        for n, h in stage.items():
            if not h["has_rule"]:
                continue
            key = (h["left"] + " | " + h["right"]).lower()
            for c in pool:
                if SequenceMatcher(None, key, c["key"]).ratio() > 0.6:
                    c["agents"].add(n); c["confidence"] = max(c["confidence"], h["confidence"]); break
            else:
                pool.append({"key": key, "h": h, "agents": {n}, "confidence": h["confidence"]})
    return sorted(pool, key=lambda c: (-len(c["agents"]), jargon(c["h"]), negated(c["h"]), -c["confidence"]))

def clearly_verified(rule, pool):
    """A free-text rule that the CV measurements verify with a clear margin (not a lucky split)."""
    pred = verify_text_rule(rule, pool)
    if not pred:
        return None
    lh = _numeric_threshold(pred, pool)[1] if pred in NUMERIC else True
    return pred if cv_rule_margin({"predicate": pred, "left_holds": lh}, pool) >= CV_MIN_MARGIN else None

## The graph (LangGraph)

In [5]:
from langgraph.graph import StateGraph, START, END

class S(TypedDict, total=False):
    bp: str; style: str; model: str
    blocks: dict; full: tuple; facts: tuple
    contrast: dict
    round1: dict; hyps: dict; rounds: int; changed: int; consensus: bool; strong: bool; judge_llm: object
    judge_dominant: str; switches: list
    rule: str; rule_nocheck: str; path: str; check_score: int; verified: bool; cv_rule: str; cv_margin: float

def agents_for(style):
    return list(ROUTES) if style == "specialist" else [f"analyst{i}" for i in range(1, 7)]

def who_focus(style, agent):
    if style == "specialist":
        if agent == "relation":
            return ("the relational specialist (the combiner) for visual pattern recognition",
                    f"You ONLY analyse {FOCUS[agent]}; you may combine size, position, containment and shape. {GUIDE[agent]}")
        return (f"the {agent} specialist for visual pattern recognition",
                f"You ONLY analyse {FOCUS[agent]}. Ignore all other dimensions. {GUIDE[agent]}")
    return ("a visual pattern analyst", "You may use every recorded property of the panels.")

def hint_for(s, agent):
    attrs = RULER_ATTRS if s["style"] == "clone" else CONTRAST_FOR[agent]
    return "\n" + contrast_summary(s["contrast"], attrs) + "\n" if attrs else ""

def perceive(s):
    full = records_block(panel_records(s["bp"], "hybrid"))                 # every fact, as the agents read it
    blocks = {a: (routed_blocks(s["bp"], a) if s["style"] == "specialist" else full) for a in agents_for(s["style"])}
    return {"blocks": blocks, "full": full, "facts": records_block(panel_records(s["bp"], "hybrid", words=True)),
            "contrast": compute_contrast(load_cv(s["bp"]), attrs=RULER_ATTRS), "rounds": 0, "changed": 0, "switches": []}

def agents(s):
    hyps = {}
    for a in agents_for(s["style"]):
        who, focus = who_focus(s["style"], a)
        left, right = s["blocks"][a]
        hyps[a] = as_answer(agent_json(AGENT_PROMPT.format(who=who, focus=focus, left=left, right=right, hint=hint_for(s, a)),
                                       model=s["model"], max_tokens=1500, temperature=0.3))
    return {"hyps": hyps, "round1": hyps}

def judge(s):
    """Original design: an LLM judge decides consensus; the local check is the fallback and audit."""
    local, dom = local_consensus(s["hyps"])
    strong = strong_override(s["hyps"])
    llm = None
    ruled = [(n, h) for n, h in s["hyps"].items() if h["has_rule"]]
    if USE_LLM_JUDGE and len(ruled) >= 2 and not strong:
        table = "\n".join(f"  {n}: {fmt_answer(h, evidence=True)}" for n, h in ruled)
        out = agent_json(JUDGE_PROMPT.format(table=table), model=s["model"], max_tokens=800)
        llm = bool(out["consensus"]) if "consensus" in out else None
        dom = str(out.get("dominant") or dom or "")
    return {"consensus": strong or (llm if llm is not None else local), "strong": strong, "judge_llm": llm,
            "judge_dominant": dom if dom in s["hyps"] else ""}

def route(s):
    if s["strong"] or s["consensus"] or s["rounds"] >= MAX_DEBATE_ROUNDS:
        return "supervisor"
    return "debate"

def debate(s):
    new, changed, log = {}, s["changed"], list(s.get("switches") or [])
    for a, own in s["hyps"].items():
        who, focus = who_focus(s["style"], a)
        left, right = s["blocks"][a]
        others = "\n".join(f"  {n}: {fmt_answer(h, evidence=True)}" for n, h in s["hyps"].items()
                           if n != a and h["has_rule"]) or "  (none)"
        raw = agent_json(DEBATE_PROMPT.format(who=who, focus=focus, own=fmt_answer(own, evidence=True), others=others,
                                              left=left, right=right, hint=hint_for(s, a)),
                         model=s["model"], max_tokens=1500, temperature=0.2)
        ans = as_answer(raw)
        cx = str(raw.get("counterexample", "") or "").strip()
        if own["has_rule"] and ans["rule"] != own["rule"]:
            kept = REQUIRE_COUNTEREXAMPLE and not cx          # a switch without a named panel is conformity: refused
            log.append({"round": s["rounds"] + 1, "agent": a, "from": own["rule"], "to": ans["rule"],
                        "counterexample": cx, "kept_own": kept})
            if kept:
                ans = own
        changed += ans["rule"] != own["rule"]
        new[a] = ans
    return {"hyps": new, "rounds": s["rounds"] + 1, "changed": changed, "switches": log}

def supervisor(s):
    """Final rule, in this order: a measured rule with a clear margin (unless confident agents' rule checks out
    instead) -> synthesis over all candidates -> a CV-verifiable candidate -> the text check -> the next candidates
    -> one look-again on the failing panels -> a narrow measured rule -> the best-scoring rule tried.
    Agent-derived rules with a negated side get one attempt to name what that side shows instead."""
    cv = load_cv(s["bp"])
    left, right = s["facts"]
    cands = candidates(s["round1"], s["hyps"])
    tried = []                                                        # (score, answer, path)

    def out(h_rule, path, score, verified, nocheck=None):
        return {"rule": h_rule, "rule_nocheck": nocheck if nocheck is not None else h_rule, "path": path,
                "check_score": score, "verified": verified,
                "cv_rule": cv_rule["nl"] if cv_rule else "", "cv_margin": round(margin, 3)}

    def check(h, path):
        score, fails = check_rule(h["left"], h["right"], left, right, s["model"])
        tried.append((score, h, path))
        return score, fails

    def finish(h, path, score, nocheck=None):
        """Ship an agent-derived rule; if a side is negated, try once to name what those panels show instead and
        keep the rewrite only if it checks at least as well."""
        if negated(h):
            named = as_answer(agent_json(NAME_PROMPT.format(L=h["left"], R=h["right"], left=left, right=right),
                                         model=s["model"], max_tokens=1000) | {"has_rule": True})
            if named["has_rule"] and named["rule"] != h["rule"] and not jargon(named):
                sc, _ = check(named, path + "+named")
                if sc >= score:
                    return out(named["rule"], path + "+named", sc, False, nocheck=nocheck if nocheck is not None else h["rule"])
        return out(h["rule"], path, score, False, nocheck=nocheck)

    cv_rule = propose_verified_rule(cv, contrast=s["contrast"])
    margin = cv_rule_margin(cv_rule, cv) if cv_rule else 0.0
    if cv_rule and margin >= CV_MIN_MARGIN:                           # 1) a clear measured rule ...
        rivals = [c for c in cands if c["confidence"] >= 0.9
                  and verify_text_rule(c["h"]["rule"], cv) != cv_rule["predicate"]][:2]
        for c in rivals:                                              #    ... unless confident agents say otherwise
            score, _ = check(c["h"], "agent-over-cv")                 #        and their rule checks out
            if score >= 11:
                return finish(c["h"], "agent-over-cv", score, nocheck=cv_rule["nl"])
        return out(cv_rule["nl"], "cv-verifier", 12, True)

    table = "\n".join(f"  [{len(c['agents'])} agree] {fmt_answer(c['h'], evidence=True)}" for c in cands) \
        or "  (no agent found a rule)"
    dom = s.get("judge_dominant") or ""
    judge_pick = f"The judge rated the {dom} agent's rule the best supported." if dom else ""
    cv_note = f"\nA measured rule splits the panels, but only narrowly: {cv_rule['nl']}" if cv_rule else ""
    sup = as_answer(agent_json(SUP_PROMPT.format(table=table, judge_pick=judge_pick,
                                                 instr=INSTR["voluntary" if s["consensus"] else "forced"],
                                                 left=left, right=right, contrast=contrast_summary(s["contrast"]),
                                                 cv_note=cv_note),
                               model=s["model"], max_tokens=1500) | {"has_rule": True})
    final = sup if sup["has_rule"] else (cands[0]["h"] if cands else sup)
    if not final["has_rule"]:
        return out(cv_rule["nl"], "cv-verifier-narrow", 12, True) if cv_rule else out("", "no-rule", 0, False)
    for h in [final] + [c["h"] for c in cands]:                       # 2) a clearly CV-verifiable candidate wins
        if clearly_verified(h["rule"], cv):
            return out(h["rule"], "search-verified", 12, True, nocheck=final["rule"])
    score, fails = check(final, "synthesis")                          # 3) check against the panel facts
    if score >= 11:
        return finish(final, "synthesis", score)
    for c in cands[:3]:                                               # 4) the most agreed-on alternatives
        if c["h"]["rule"] != final["rule"]:
            sc, _ = check(c["h"], "checked-alternative")
            if sc >= 11:
                return finish(c["h"], "checked-alternative", sc, nocheck=final["rule"])
    again = as_answer(agent_json(LOOK_AGAIN_PROMPT.format(L=final["left"], R=final["right"], fails="\n".join(fails),
                                                          left=left, right=right),
                                 model=s["model"], max_tokens=1500) | {"has_rule": True})
    if again["has_rule"]:                                             # 5) one look-again on the failing panels
        sc, _ = check(again, "look-again")
        if sc >= 11:
            return finish(again, "look-again", sc, nocheck=final["rule"])
    if cv_rule:                                                       # 6) a narrow but exact measured split
        return out(cv_rule["nl"], "cv-verifier-narrow", 12, True, nocheck=final["rule"])
    clean = [t for t in tried if not jargon(t[1])] or tried          # 7) the best-scoring rule tried (no jargon;
    best_score, best, _ = max(clean, key=lambda t: t[0])              #    ties go to the supervisor's, tried first)
    return finish(best, "unchecked-best", best_score, nocheck=final["rule"])

g = StateGraph(S)
for name, fn in [("perceive", perceive), ("agents", agents), ("judge", judge), ("debate", debate), ("supervisor", supervisor)]:
    g.add_node(name, fn)
g.add_edge(START, "perceive"); g.add_edge("perceive", "agents"); g.add_edge("agents", "judge")
g.add_conditional_edges("judge", route, {"debate": "debate", "supervisor": "supervisor"})
g.add_edge("debate", "judge"); g.add_edge("supervisor", END)
GRAPH = g.compile()
print("graph compiled")

graph compiled


## Run every arm (resumable)

In [6]:
METER.tag = "perception: 1 VLM call per problem"
for bp in PROBLEMS:
    vlm_fields(bp)                                     # cached if 01 already ran

for style, model in RUNS:
    arm = f"pipeline__{style}__{model.split('/')[-1]}"
    rows = load_predictions(arm)
    METER.tag = arm
    print(f"\n=== {arm} ===")
    for bp in PROBLEMS:
        if bp in rows and not RERUN:
            continue
        before, calls0 = METER.total, METER.calls
        try:
            out = GRAPH.invoke({"bp": bp, "style": style, "model": model})
        except BudgetExceeded:
            raise
        except Exception as e:
            print(f"  {bp}  FAILED ({type(e).__name__}: {str(e)[:100]}) — skipped, re-run to retry")
            continue
        rows[bp] = {"bp": bp, "rule": out["rule"], "rule_nocheck": out["rule_nocheck"], "path": out["path"],
                    "check_score": out["check_score"], "verified": out["verified"], "rounds": out["rounds"],
                    "consensus": out["consensus"], "strong_consensus": out["strong"], "judge_llm": out.get("judge_llm"),
                    "judge_dominant": out.get("judge_dominant", ""), "switches": out.get("switches", []),
                    "cv_rule": out.get("cv_rule", ""), "cv_margin": out.get("cv_margin", 0.0),
                    "round1": out["round1"], "final_hyps": out["hyps"], "calls": METER.calls - calls0,
                    "cost_usd": METER.total - before, "perception_cost_usd": perception_cost(bp)}
        save_predictions(arm, rows)
        print(f"  {bp}  [{out['path']:19s}] check {out['check_score']:2d}/12  {out['rule'][:95]}")
METER.report()


=== pipeline__specialist__gpt-oss-120b ===


  bp0002  [cv-verifier        ] check 12/12  Left = large figures, Right = small figures


  bp0004  [cv-verifier        ] check 12/12  Left = convex figures, Right = nonconvex figures


  bp0007  [agent-over-cv      ] check 12/12  Left = vertically elongated figures, Right = horizontally elongated figures


  bp0010  [unchecked-best     ] check 10/12  Left = shapes with an odd number of corners, Right = shapes with an even number of corners


...

  bp0033  [synthesis+named    ] check 12/12  Left = figures with acute corners, Right = figures lacking acute corners


  bp0038  [synthesis          ] check 12/12  Left = triangle larger than circle, Right = circle larger than triangle


  bp0046  [synthesis          ] check 11/12  Left = triangle's spatial relation to the circle, Right = circle's spatial relation to the tria


  bp0057  [unchecked-best     ] check  9/12  Left = two matching shapes of equal size, Right = shapes differ in type or size


  bp0096  [unchecked-best+named] check 10/12  Left = upright triangle-shaped polygons, Right = non‑upright or non‑triangle polygons
--- spend: $0.3497 of $0.60 ceiling, 373 calls ---
   pipeline__specialist__gpt-oss-120b           $0.3497


## Trace one problem (every agent's answer, for debugging)

In [7]:
arm = f"pipeline__{RUNS[0][0]}__{RUNS[0][1].split('/')[-1]}"
bp = "bp0038" if "bp0038" in PROBLEMS else PROBLEMS[0]
r = load_predictions(arm)[bp]
print(f"{bp}   answer: {gt(bp)}\n")
for n, h in r["round1"].items():
    print(f"  round 1  {n:9s} {fmt_answer(h)}")
print(f"\n  consensus={r['consensus']} (LLM judge: {r['judge_llm']}, strong override: {r['strong_consensus']})  debate rounds={r['rounds']}  calls={r['calls']}")
for n, h in r["final_hyps"].items():
    print(f"  final    {n:9s} {fmt_answer(h)}")
for sw in r.get("switches", []):
    print(f"  debate switch {sw['agent']}: {'REFUSED (no counterexample)' if sw['kept_own'] else 'accepted, ' + sw['counterexample']}")
print(f"\n  supervisor path: {r['path']}   check {r['check_score']}/12   CV-verified: {r['verified']}")
if r.get("cv_rule"):
    print(f"  measured rule: {r['cv_rule']}  (margin {r['cv_margin']:.2f}; clear if >= {CV_MIN_MARGIN})")
print(f"  rule without check: {r['rule_nocheck']}\n  final rule        : {r['rule']}")

bp0038   answer: Left = Triangle larger than circle, Right = Triangle smaller than circle

  round 1  shape     no rule
  round 1  topology  no rule
  round 1  quantity  no rule
  round 1  spatial   Left = triangle points toward the circle, Right = triangle points away from the circle (confidence 0.90)
  round 1  size      Left = triangle larger than circle, Right = circle larger than triangle (confidence 1.00)
  round 1  relation  Left = triangle larger than circle, Right = circle larger than triangle (confidence 1.00)

  consensus=False (LLM judge: False, strong override: False)  debate rounds=1  calls=19
  final    shape     no rule
  final    topology  no rule
  final    quantity  no rule
  final    spatial   Left = triangle points toward the circle, Right = triangle points away from the circle (confidence 0.90)
  final    size      Left = triangle larger than circle, Right = circle larger than triangle (confidence 1.00)
  final    relation  Left = triangle larger than circle, Righ